In [1]:
import os
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc, classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

import xgboost as xgb
import joblib

IMAGE_DIR = 'C:/Users/RTX2080Ti/Desktop/AD Connectivity/Images/Eyes Open'
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 10
RANDOM_STATE = 42
TEST_SIZE = 0.30
N_FOLDS = 5
SAVE_DIR = 'models_and_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)

def gather_image_paths(image_dir):
    classes = sorted([d for d in os.listdir(image_dir) if os.path.isdir(os.path.join(image_dir, d))])
    paths, labels = [], []
    for cls in classes:
        for ext in ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif'):
            for p in glob(os.path.join(image_dir, cls, ext)):
                paths.append(p)
                labels.append(cls)
    df = pd.DataFrame({'path': paths, 'label': labels})
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return df

def load_images(paths, img_size):
    X = np.zeros((len(paths), img_size[0], img_size[1], 3), dtype=np.float32)
    for i, path in enumerate(tqdm(paths, desc="Loading images")):
        img = load_img(path, target_size=img_size)
        X[i] = img_to_array(img) / 255.0
    return X

def build_custom_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(128, activation='relu', name='feature_layer'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.0005),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def main():
    print("Gathering dataset...")
    df = gather_image_paths(IMAGE_DIR)
    print(f"Found {len(df)} images across {df.label.nunique()} classes: {df.label.unique()}")

    le = LabelEncoder()
    df['label_enc'] = le.fit_transform(df['label'])
    n_classes = len(le.classes_)
    joblib.dump(le, os.path.join(SAVE_DIR, 'label_encoder.joblib'))

    # === 70–30 TRAIN–TEST SPLIT ===
    X_paths = df['path'].values
    y = df['label_enc'].values
    X_train_paths, X_test_paths, y_train, y_test = train_test_split(
        X_paths, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    print(f"Train: {len(X_train_paths)}, Test: {len(X_test_paths)}")

    # === LOAD AND TRAIN CUSTOM CNN ===
    X_train = load_images(X_train_paths, IMG_SIZE)
    X_test = load_images(X_test_paths, IMG_SIZE)

    cnn = build_custom_cnn((IMG_SIZE[0], IMG_SIZE[1], 3), n_classes)
    history = cnn.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
                      validation_split=0.2, verbose=1)

    # Plot training and validation loss curves
    plt.figure(figsize=(8, 6))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(SAVE_DIR, 'loss_curves.png'))
    plt.close()  # Close the figure to free memory

    cnn.save(os.path.join(SAVE_DIR, 'custom_cnn.h5'))

    # === FEATURE EXTRACTION ===
    feature_extractor = Model(inputs=cnn.inputs, outputs=cnn.get_layer('feature_layer').output)
    X_train_feats = feature_extractor.predict(X_train)
    X_test_feats = feature_extractor.predict(X_test)

    print("Feature shapes:", X_train_feats.shape, X_test_feats.shape)

    # === SCALE FEATURES ===
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_feats)
    X_test_scaled = scaler.transform(X_test_feats)
    joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler.joblib'))

    # === 5-FOLD CROSS VALIDATION WITH XGBOOST ===
    print(f"Starting {N_FOLDS}-fold CV on CNN features with XGBoost...")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    cv_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': []}

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train), 1):
        print(f"Fold {fold}")
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        clf = xgb.XGBClassifier(
            eval_metric='mlogloss',  # Use 'mlogloss' for multiclass; 'logloss' is alias but explicit is better
            random_state=RANDOM_STATE,
            early_stopping_rounds=10  # Moved here: stop early if no improvement for 10 rounds (saves time if converging fast)
        )
        clf.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],  # Add this: use the val fold for evaluation during training
            verbose=10,  # Add this: print eval metrics every 10 rounds (set to True for every round, or an int for interval)
        )

        y_pred = clf.predict(X_val)
        y_proba = clf.predict_proba(X_val)

        acc = accuracy_score(y_val, y_pred)
        prec = precision_score(y_val, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_val, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)

        try:
            y_val_b = label_binarize(y_val, classes=np.arange(n_classes))
            roc_auc = roc_auc_score(y_val_b, y_proba, average='macro', multi_class='ovr')
        except:
            roc_auc = np.nan

        print(f" acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f} auc={roc_auc:.4f}")
        cv_results['accuracy'].append(acc)
        cv_results['precision'].append(prec)
        cv_results['recall'].append(rec)
        cv_results['f1'].append(f1)
        cv_results['roc_auc'].append(roc_auc)

    print("\\nCV mean ± std:")
    for k, v in cv_results.items():
        print(f" {k}: {np.nanmean(v):.4f} ± {np.nanstd(v):.4f}")

    # === FINAL TRAIN ON FULL TRAIN SET, TEST ON 30% HOLD-OUT ===
    final_clf = xgb.XGBClassifier(eval_metric='mlogloss', random_state=RANDOM_STATE)  # Updated to remove deprecated use_label_encoder and use 'mlogloss' consistently
    final_clf.fit(X_train_scaled, y_train)
    joblib.dump(final_clf, os.path.join(SAVE_DIR, 'final_xgb_model.joblib'))

    y_test_pred = final_clf.predict(X_test_scaled)
    y_test_proba = final_clf.predict_proba(X_test_scaled)

    print("\\n--- HOLD-OUT TEST RESULTS ---")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("Precision:", precision_score(y_test, y_test_pred, average='macro', zero_division=0))
    print("Recall:", recall_score(y_test, y_test_pred, average='macro', zero_division=0))
    print("F1-score:", f1_score(y_test, y_test_pred, average='macro', zero_division=0))
    print("\\nClassification Report:")
    print(classification_report(y_test, y_test_pred, target_names=le.classes_))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(cmap=plt.cm.Blues, values_format='d')
    plt.title('Confusion Matrix')
    plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'))
    plt.close()

    try:
        y_test_b = label_binarize(y_test, classes=np.arange(n_classes))
        roc_auc = roc_auc_score(y_test_b, y_test_proba, average='macro', multi_class='ovr')
        print("ROC AUC (ovr):", roc_auc)

        # ROC-AUC Curves (One-vs-Rest for multi-class)
        fpr = dict()
        tpr = dict()
        roc_auc_dict = dict()  # To store per-class AUC

        plt.figure(figsize=(8, 6))
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test_b[:, i], y_test_proba[:, i])
            roc_auc_dict[i] = auc(fpr[i], tpr[i])
            plt.plot(fpr[i], tpr[i], label=f'{le.classes_[i]} (AUC = {roc_auc_dict[i]:.2f})')

        plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC-AUC Curves (One-vs-Rest)')
        plt.legend(loc='lower right')
        plt.grid(True)
        plt.savefig(os.path.join(SAVE_DIR, 'roc_curves.png'))
        plt.close()
    except:
        pass

    print("\\nAll done! CNN + XGBoost pipeline complete.")

if __name__ == "__main__":
    main()

Gathering dataset...
Found 7520 images across 3 classes: ['AD' 'Control' 'FTD']
Train: 5264, Test: 2256


Loading images: 100%|█████████████████████████████████████████████████████████████| 2256/2256 [00:02<00:00, 811.54it/s]
C:\Users\RTX2080Ti\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 23s 162ms/step - accuracy: 0.4495 - loss: 1.0479 - val_accuracy: 0.6543 - val_loss: 0.8153
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 17s 130ms/step - accuracy: 0.6671 - loss: 0.7847 - val_accuracy: 0.7806 - val_loss: 0.5674
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 18s 136ms/step - accuracy: 0.7697 - loss: 0.5490 - val_accuracy: 0.8329 - val_loss: 0.4387
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 18s 135ms/step - accuracy: 0.8188 - loss: 0.4620 - val_accuracy: 0.8585 - val_loss: 0.3727
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 18s 138ms/step - accuracy: 0.8514 - loss: 0.3819 - val_accuracy: 0.8889 - val_loss: 0.3295
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 18s 140ms/step - accuracy: 0.8927 - loss: 0.3059 - val_accuracy: 0.8955 - val_loss: 0.3033
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 19s 141ms/step - accuracy: 0.8905 - loss: 0.2924 - val_accuracy: 0.8689 - val_loss: 0.3419
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 20s 151ms/step - accuracy: 0.9041 - loss: 0

  3/165 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step  

C:\Users\RTX2080Ti\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor']. Received: the structure of inputs=*
  warnings.warn(


165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step
Feature shapes: (5264, 128) (2256, 128)
Starting 5-fold CV on CNN features with XGBoost...
Fold 1
[0]	validation_0-mlogloss:0.75930
[10]	validation_0-mlogloss:0.13339
[20]	validation_0-mlogloss:0.10591
[30]	validation_0-mlogloss:0.10357
[40]	validation_0-mlogloss:0.10644
[42]	validation_0-mlogloss:0.10749
 acc=0.9649 prec=0.9631 rec=0.9625 f1=0.9628 auc=0.9965
Fold 2
[0]	validation_0-mlogloss:0.75475
[10]	validation_0-mlogloss:0.13287
[20]	validation_0-mlogloss:0.10386
[30]	validation_0-mlogloss:0.10429
 acc=0.9611 prec=0.9574 rec=0.9580 f1=0.9576 auc=0.9973
Fold 3
[0]	validation_0-mlogloss:0.76195
[10]	validation_0-mlogloss:0.14275
[20]	validation_0-mlogloss:0.11077
[30]	validation_0-mlogloss:0.10990
[35]	validation_0-mlogloss:0.11000
 acc=0.9592 prec=0.9526 rec=0.9596 f1=0.9558 auc=0.9960
Fold 4
[0]	validation_0-mlogloss:0.75632
[10]	validation_0-mlogloss:0.13129
[20]	validation_0-mlogloss:0.09538
[30]	